# Logistic Regression From Scratch (Mathematical)

Implement logistic regression from scratch using only NumPy — understand the math behind the algorithm.

## 1. Import Libraries

In [22]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

## 2. The Math Behind Logistic Regression

**Sigmoid Function:**
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Hypothesis:**
$$h(x) = \sigma(w^T x + b)$$

**Cost Function (Log Loss / Cross-Entropy):**
$$J(w, b) = -\frac{1}{m} \sum_{i=1}^{m} [y_i \log(h(x_i)) + (1-y_i) \log(1-h(x_i))]$$

**Gradient Descent:**
$$\frac{\partial J}{\partial w} = \frac{1}{m} X^T (h(X) - y)$$
$$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (h(x_i) - y_i)$$

## 3. Logistic Regression Class

In [23]:
class LogisticRegression:
    def __init__(self, learning_rate=0.01, iterations=1000):
        self.learning_rate = learning_rate
        self.iterations = iterations
        self.weights = None
        self.bias = None
        self.cost_history = []
    
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))
    
    def fit(self, X, y):
        m, n = X.shape
        self.weights = np.zeros(n)
        self.bias = 0
        
        for iteration in range(self.iterations):
            # Forward pass
            z = np.dot(X, self.weights) + self.bias
            predictions = self.sigmoid(z)
            
            # Compute cost (log loss)
            cost = -np.mean(y * np.log(predictions + 1e-15) + 
                           (1 - y) * np.log(1 - predictions + 1e-15))
            self.cost_history.append(cost)
            
            # Compute gradients
            dw = np.dot(X.T, (predictions - y)) / m
            db = np.mean(predictions - y)
            
            # Update parameters
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            
            if (iteration + 1) % 100 == 0:
                print(f"Iteration {iteration + 1}/{self.iterations}, Cost: {cost:.4f}")
        
        return self
    
    def predict_proba(self, X):
        z = np.dot(X, self.weights) + self.bias
        return self.sigmoid(z)
    
    def predict(self, X, threshold=0.5):
        proba = self.predict_proba(X)
        return (proba >= threshold).astype(int)
    
    def score(self, X, y):
        predictions = self.predict(X)
        accuracy = np.mean(predictions == y)
        return accuracy

## 4. Load and Prepare Data

In [24]:
# Load exam pass/fail data
df = pd.read_csv('exam_pass.csv')
print(df.head())
print(f"Shape: {df.shape}")
print(f"Data types:\n{df.dtypes}")

   hours_studied  prior_score  passed
0           4.36         65.1       1
1           0.26         65.4       0
2           5.50         45.3       1
3           4.35         30.3       1
4           4.20         59.2       1
Shape: (200, 3)
Data types:
hours_studied    float64
prior_score      float64
passed             int64
dtype: object


In [25]:
# Prepare features and target
X = df[['hours_studied', 'sleep_hours']].values
y = df['passed'].values

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Class distribution: {np.bincount(y)}")

KeyError: "['sleep_hours'] not in index"

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## 5. Train the Model

In [ ]:
# Create and train model
model = LogisticRegression(learning_rate=0.1, iterations=500)
model.fit(X_train, y_train)

## 6. Evaluate the Model

In [ ]:
# Predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Accuracy
train_accuracy = model.score(X_train, y_train)
test_accuracy = model.score(X_test, y_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Confusion Matrix
from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_test, y_test_pred)
print(f"\nConfusion Matrix:\n{cm}")
print(f"\nClassification Report:\n{classification_report(y_test, y_test_pred)}")

## 7. Visualize Cost Function and Decision Boundary

In [ ]:
# Plot cost function
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(model.cost_history)
plt.xlabel('Iteration')
plt.ylabel('Cost (Log Loss)')
plt.title('Cost Function Over Iterations')
plt.grid(True)

# Plot predictions and data
plt.subplot(1, 2, 2)
plt.scatter(X_test[y_test == 0, 0], X_test[y_test == 0, 1], label='Failed (0)', alpha=0.6)
plt.scatter(X_test[y_test == 1, 0], X_test[y_test == 1, 1], label='Passed (1)', alpha=0.6)
plt.xlabel('Hours Studied (Standardized)')
plt.ylabel('Sleep Hours (Standardized)')
plt.title('Test Data with Predictions')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 8. Make Predictions on New Data

In [ ]:
# Predict for new student
new_student = np.array([[5, 7]])  # 5 hours studied, 7 hours sleep
new_student_scaled = scaler.transform(new_student)

proba = model.predict_proba(new_student_scaled)[0]
prediction = model.predict(new_student_scaled)[0]

print(f"Prediction Probability: {proba:.4f}")
print(f"Prediction: {'Pass' if prediction == 1 else 'Fail'}")

## Key Takeaways

- **Sigmoid function** converts linear output to probabilities (0-1)
- **Cost function** (log loss) measures prediction error
- **Gradient descent** updates weights to minimize cost
- **Standardization** helps the model converge faster
- **Decision boundary** separates classes at probability = 0.5

## Cheat Sheet

### Quick Reference

| Concept | Formula / Code |
|---|---|
| **Sigmoid** | `1 / (1 + np.exp(-z))` |
| **Hypothesis** | `h(x) = sigmoid(w^T x + b)` |
| **Cost (Log Loss)** | `-mean(y*log(h) + (1-y)*log(1-h))` |
| **Gradient (weights)** | `dw = X^T(h - y) / m` |
| **Gradient (bias)** | `db = mean(h - y)` |
| **Update weights** | `w = w - learning_rate * dw` |
| **Prediction** | `argmax(predict_proba(X))` |
| **Accuracy** | `mean(predictions == y)` |

### Model Parameters

```python
LogisticRegression(
    learning_rate=0.01,      # Step size for gradient descent
    iterations=1000          # Number of training iterations
)
```

### Key Methods

```python
model.fit(X, y)              # Train the model
model.predict(X)             # Get class predictions
model.predict_proba(X)       # Get probabilities
model.score(X, y)            # Get accuracy
```

### When to Use Logistic Regression

✅ **Good for:**
- Binary classification problems
- When you need probability estimates
- When interpretability is important
- Fast training and prediction

❌ **Not ideal for:**
- Non-linearly separable data
- Complex decision boundaries
- High-dimensional data (use regularization)